torch: Main PyTorch library for tensor operations and GPU acceleration.

nn: Provides modules and classes for building neural networks.

DataLoader: Helps load large datasets in mini-batches, optionally shuffling and parallelizing them.

datasets: Gives access to popular datasets like FashionMNIST.

ToTensor(): Converts image data to PyTorch tensors (scales pixel values from 0–255 to 0–1).

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

downloads the training split of FashionMNIST to "data" directory.

transform=ToTensor() converts PIL images to PyTorch tensors.



In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 207kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.85MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.4MB/s]


In [ ]:
# Number of samples processed before the model updates.
batch_size = 64

# Create data loaders.
# Wraps datasets into iterable mini-batches of 64 images.
# Defaults to shuffle=False. You can add shuffle=True for training.

# Note: If you set shuffle=True, the data will be shuffled at every epoch.

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)


# Loads the first batch of the test set.

# Prints the shape:

# X: [Batch size, Channels, Height, Width] → [64, 1, 28, 28].

# y: Labels as 1D tensor of shape [64].
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


[64, 1, 28, 28] → Batch of 64 grayscale images.

64: Batch size (N).

1: Channel (C), since grayscale images have 1 channel (RGB would have 3).

28, 28: Height (H) and Width (W) of each image (e.g., MNIST digits).

y (Labels)

[64] → 1D tensor of 64 labels (one per image in the batch).

torch.int64: Labels are integers (e.g., class indices like 0 for "digit 0").

#### Why dataloaders used?

- **Batching**: Splits data into batches for mini-batch training.  
- **Shuffling**: Randomizes data order to prevent overfitting.  
- **Parallel Loading**: Uses multiple workers (`num_workers`) for faster I/O.  
- **Memory Efficiency**: Loads data on-demand instead of all at once.  
- **GPU Transfer**: Easily moves batches to GPU with `.to(device)`.  
- **Custom Datasets**: Works with `Dataset` class for flexible data handling.  
- **Auto-Padding**: Handles variable-length sequences (e.g., NLP) via `collate_fn`.  
- **Simplified Loops**: Clean iteration over batches in training/eval.  

In [ ]:
# Tries to use a GPU or TPU if available, otherwise defaults to CPU.    
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() #nn.Flatten() reshapes [1, 28, 28] into [784].
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device) #Instantiates the model and moves it to GPU (or CPU).
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
loss_fn = nn.CrossEntropyLoss() # Defines the loss function for multi-class classification.
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset) # Gets the total number of samples in the dataset.
    model.train() # Sets the model to training mode.
    for batch, (X, y) in enumerate(dataloader): 
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset) # Gets the total number of samples in the dataset.
    num_batches = len(dataloader) # Gets the number of batches in the dataloader.
    model.eval() # Sets the model to evaluation mode.
    test_loss, correct = 0, 0 # Initializes test loss and correct predictions.
    with torch.no_grad(): # Disables gradient calculation for inference to save memory and computation.
        for X, y in dataloader:
            X, y = X.to(device), y.to(device) # Moves data to the appropriate device (GPU or CPU).
            pred = model(X) # Makes predictions using the model.
            test_loss += loss_fn(pred, y).item()    # Accumulates the loss for the batch.
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() # Counts the number of correct predictions.
    test_loss /= num_batches
    correct /= size # Calculates the accuracy as a fraction of correct predictions over total samples.
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [8]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301620  [   64/60000]
loss: 2.292605  [ 6464/60000]
loss: 2.277963  [12864/60000]
loss: 2.272280  [19264/60000]
loss: 2.255783  [25664/60000]
loss: 2.223477  [32064/60000]
loss: 2.238332  [38464/60000]
loss: 2.197888  [44864/60000]
loss: 2.192132  [51264/60000]
loss: 2.160119  [57664/60000]
Test Error: 
 Accuracy: 39.8%, Avg loss: 2.158960 

Epoch 2
-------------------------------
loss: 2.164738  [   64/60000]
loss: 2.158794  [ 6464/60000]
loss: 2.108056  [12864/60000]
loss: 2.127187  [19264/60000]
loss: 2.077066  [25664/60000]
loss: 2.015080  [32064/60000]
loss: 2.058613  [38464/60000]
loss: 1.968355  [44864/60000]
loss: 1.968340  [51264/60000]
loss: 1.909540  [57664/60000]
Test Error: 
 Accuracy: 56.7%, Avg loss: 1.903167 

Epoch 3
-------------------------------
loss: 1.926285  [   64/60000]
loss: 1.899143  [ 6464/60000]
loss: 1.794763  [12864/60000]
loss: 1.841890  [19264/60000]
loss: 1.733544  [25664/60000]
loss: 1.676380  [32064/600

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1] # Gets the first sample from the test dataset.
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Coat", Actual: "Ankle boot"
